# Semantic Segmentation

## Imports

In [37]:
import numpy as np
import pandas as pd

from PIL import Image
import matplotlib.pyplot as plt
from skimage import io, color, filters
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import os
from glob import glob
import warnings
warnings.filterwarnings('ignore')

print('Imported')

Imported


## Configurations

In [15]:
class Config:
    
    IMAGE_DIR = '../raw_images/'  # Your satellite images folder
    OUTPUT_DIR = '../processed/images/'
    
    # Sampling (to reduce computation)
    SAMPLE_RATE = 0.1  # Use 10% of pixels for training (reduces memory)
    
    # Classes
    CLASSES = {
        0: 'ice',      # Glacier ice
        1: 'water',    # Ocean/meltwater
        2: 'other'     # Rock, shadow, etc.
    }
    
    # Model
    N_ESTIMATORS = 100  # Reduced for speed
    MAX_DEPTH = 15
    MIN_SAMPLES_SPLIT = 10
    
    # Feature window size
    WINDOW_SIZE = 5  # 5x5 neighborhood features

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

print("LIGHTWEIGHT GLACIER SEGMENTATION")
print()
print(f"Sample rate: {config.SAMPLE_RATE*100}% of pixels")
print(f"Classes: {list(config.CLASSES.values())}")
print(f"Window size: {config.WINDOW_SIZE}x{config.WINDOW_SIZE}")


LIGHTWEIGHT GLACIER SEGMENTATION

Sample rate: 10.0% of pixels
Classes: ['ice', 'water', 'other']
Window size: 5x5


## Step 1: Pixel Feature Extraction

In [7]:
def extract_pixel_features(image, sample_pixels=None):
    
    # Normalization
    if image.dtype == np.uint8:
        image = image.astype(float) / 255.0
    
    h, w, c = image.shape
    
    # Computation
    gray = color.rgb2gray(image)
    gradient = filters.sobel(gray)
    
    # Pad image for window operations
    pad = config.WINDOW_SIZE // 2
    image_padded = np.pad(image, ((pad, pad), (pad, pad), (0, 0)), mode='reflect')
    gray_padded = np.pad(gray, ((pad, pad), (pad, pad)), mode='reflect')
    
    # If sampling, select random pixels
    if sample_pixels is not None:
        pixel_indices = sample_pixels
    else:
        # Use all pixels
        pixel_indices = np.array([[i, j] for i in range(h) for j in range(w)])
    
    n_pixels = len(pixel_indices)
    features = np.zeros((n_pixels, 13))
    
    for idx, (i, j) in enumerate(pixel_indices):
        # Get window
        window_rgb = image_padded[i:i+config.WINDOW_SIZE, j:j+config.WINDOW_SIZE]
        window_gray = gray_padded[i:i+config.WINDOW_SIZE, j:j+config.WINDOW_SIZE]
        
        # RGB values
        features[idx, 0] = image[i, j, 0]  # Red
        features[idx, 1] = image[i, j, 1]  # Green
        features[idx, 2] = image[i, j, 2]  # Blue
        
        # Neighborhood RGB mean
        features[idx, 3] = window_rgb[:, :, 0].mean()  # Red mean
        features[idx, 4] = window_rgb[:, :, 1].mean()  # Green mean
        features[idx, 5] = window_rgb[:, :, 2].mean()  # Blue mean
        
        # Neighborhood RGB std
        features[idx, 6] = window_rgb[:, :, 0].std()   # Red std
        features[idx, 7] = window_rgb[:, :, 1].std()   # Green std
        features[idx, 8] = window_rgb[:, :, 2].std()   # Blue std
        
        # Gradient magnitude
        features[idx, 9]

## Step 2: Labeling

In [23]:
def create_manual_labels_interactive(image_path):
    """
    Helper to manually label a few images for training
    Click to label pixels: 
    - Left click = ice (red)
    - Right click = water (blue)
    - Middle click = other (green)
    """
    img = np.array(Image.open(image_path))
    labels = np.zeros(img.shape[:2], dtype=np.uint8)  # 0=unlabeled
    
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    ax[0].imshow(img)
    ax[0].set_title('Original - Click to label pixels')
    ax[1].imshow(labels, cmap='tab10')
    ax[1].set_title('Labels: Red=Ice, Blue=Water, Green=Other')
    
    coords = []
    label_vals = []
    
    def onclick(event):
        if event.inaxes == ax[0]:
            x, y = int(event.xdata), int(event.ydata)
            if event.button == 1:  # Left click = ice
                labels[y, x] = 1
                color = 'red'
            elif event.button == 3:  # Right click = water
                labels[y, x] = 2
                color = 'blue'
            elif event.button == 2:  # Middle = other
                labels[y, x] = 3
                color = 'yellow'
            
            ax[0].plot(x, y, 'o', color=color, markersize=3)
            ax[1].imshow(labels, cmap='tab10')
            plt.draw()
            
            coords.append([y, x])
            label_vals.append(labels[y, x])
    
    fig.canvas.mpl_connect('button_press_event', onclick)
    plt.show()
    
    return labels, np.array(coords), np.array(label_vals)


## Step 3: Simple Feature Extration

In [ ]:
def extract_features_fast(image):
    """
    Extract 9 simple features per pixel:
    1-3: RGB values
    4-6: RGB means in 7x7 window
    7: Brightness
    8-9: Blue-Red ratio, Green-Red ratio (water indices)
    """
    if isinstance(image, str):
        image = np.array(Image.open(image))
    
    # Ensure RGB
    if len(image.shape) == 2:
        image = np.stack([image]*3, axis=-1)
    
    h, w = image.shape[:2]
    
    # Normalize to 0-1
    img_norm = image.astype(float) / 255.0
    
    # Create feature array
    features = np.zeros((h, w, 9))
    
    # RGB values
    features[:, :, 0] = img_norm[:, :, 0]  # Red
    features[:, :, 1] = img_norm[:, :, 1]  # Green
    features[:, :, 2] = img_norm[:, :, 2]  # Blue
    
    # Local means (smoothed RGB)
    from scipy.ndimage import uniform_filter
    features[:, :, 3] = uniform_filter(img_norm[:, :, 0], size=7)  # Red mean
    features[:, :, 4] = uniform_filter(img_norm[:, :, 1], size=7)  # Green mean
    features[:, :, 5] = uniform_filter(img_norm[:, :, 2], size=7)  # Blue mean
    
    # Brightness
    features[:, :, 6] = img_norm.mean(axis=2)
    
    # Color ratios (water detection)
    with np.errstate(divide='ignore', invalid='ignore'):
        features[:, :, 7] = np.where(img_norm[:, :, 0] > 0, 
                                     img_norm[:, :, 2] / img_norm[:, :, 0], 0)  # B/R
        features[:, :, 8] = np.where(img_norm[:, :, 0] > 0,
                                     img_norm[:, :, 1] / img_norm[:, :, 0], 0)  # G/R
    
    return features

## Step 4: Simplified Labeling (Rule-based for quick start)

In [ ]:
def extract_features_fast(image):
    """
    Extract 9 simple features per pixel:
    1-3: RGB values
    4-6: RGB means in 7x7 window
    7: Brightness
    8-9: Blue-Red ratio, Green-Red ratio (water indices)
    """
    if isinstance(image, str):
        image = np.array(Image.open(image))
    
    # Ensure RGB
    if len(image.shape) == 2:
        image = np.stack([image]*3, axis=-1)
    
    h, w = image.shape[:2]
    
    # Normalize to 0-1
    img_norm = image.astype(float) / 255.0
    
    # Create feature array
    features = np.zeros((h, w, 9))
    
    # RGB values
    features[:, :, 0] = img_norm[:, :, 0]  # Red
    features[:, :, 1] = img_norm[:, :, 1]  # Green
    features[:, :, 2] = img_norm[:, :, 2]  # Blue
    
    # Local means (smoothed RGB)
    from scipy.ndimage import uniform_filter
    features[:, :, 3] = uniform_filter(img_norm[:, :, 0], size=7)  # Red mean
    features[:, :, 4] = uniform_filter(img_norm[:, :, 1], size=7)  # Green mean
    features[:, :, 5] = uniform_filter(img_norm[:, :, 2], size=7)  # Blue mean
    
    # Brightness
    features[:, :, 6] = img_norm.mean(axis=2)
    
    # Color ratios (water detection)
    with np.errstate(divide='ignore', invalid='ignore'):
        features[:, :, 7] = np.where(img_norm[:, :, 0] > 0, 
                                     img_norm[:, :, 2] / img_norm[:, :, 0], 0)  # B/R
        features[:, :, 8] = np.where(img_norm[:, :, 0] > 0,
                                     img_norm[:, :, 1] / img_norm[:, :, 0], 0)  # G/R
    
    return features


## Step 5: Creating Labels (Unsupervised)

In [26]:
def create_simple_labels(image):
    """
    Create simple labels using thresholds (for initial training)
    You can replace this with manual labels later
    
    Rules:
    - Ice: High brightness (>0.7), low blue/red ratio
    - Water: Low brightness (<0.4), high blue/red ratio  
    - Other: Everything else
    """
    if isinstance(image, str):
        image = np.array(Image.open(image))
    
    img_norm = image.astype(float) / 255.0
    
    # Calculate indices
    brightness = img_norm.mean(axis=2)
    
    with np.errstate(divide='ignore', invalid='ignore'):
        br_ratio = np.where(img_norm[:, :, 0] > 0.01,
                           img_norm[:, :, 2] / img_norm[:, :, 0], 1.0)
    
    # Initialize labels
    labels = np.zeros(image.shape[:2], dtype=np.uint8)
    
    # Ice: bright and low blue/red
    labels[(brightness > 0.65) & (br_ratio < 1.1)] = 1
    
    # Water: dark and high blue (or very dark)
    labels[(brightness < 0.35) | ((brightness < 0.5) & (br_ratio > 1.2))] = 2
    
    # Other: everything else
    labels[labels == 0] = 3
    
    return labels


In [28]:
def train_segmentation_model(image_paths, sample_size=10000):
    """
    Train model on sampled pixels from multiple images
    
    Args:
        image_paths: List of image file paths
        sample_size: Number of pixels to sample per image
    """
    
    print("="*80)
    print("TRAINING SEGMENTATION MODEL")
    print("="*80)
    
    X_all = []
    y_all = []
    
    # Process each image
    for i, img_path in enumerate(image_paths[:10]):  # Use first 10 images for training
        print(f"Processing {i+1}/10: {os.path.basename(img_path)}")
        
        # Load image
        image = np.array(Image.open(img_path))
        
        # Extract features
        features = extract_features_fast(image)
        h, w, n_features = features.shape
        
        # Create labels (using simple rules - replace with manual labels if available)
        labels = create_simple_labels(image)
        
        # Reshape to (pixels, features)
        X = features.reshape(-1, n_features)
        y = labels.reshape(-1)
        
        # Sample pixels to reduce memory
        n_pixels = h * w
        if n_pixels > sample_size:
            indices = np.random.choice(n_pixels, sample_size, replace=False)
            X = X[indices]
            y = y[indices]
        
        # Remove unlabeled pixels
        mask = y > 0
        X = X[mask]
        y = y[mask]
        
        X_all.append(X)
        y_all.append(y)
    
    # Combine all samples
    X_train = np.vstack(X_all)
    y_train = np.hstack(y_all)
    
    print(f"\nTotal training samples: {len(X_train):,}")
    print(f"Class distribution:")
    unique, counts = np.unique(y_train, return_counts=True)
    for cls, count in zip(unique, counts):
        cls_name = ['unlabeled', 'ice', 'water', 'other'][cls]
        print(f"  {cls_name}: {count:,} ({count/len(y_train)*100:.1f}%)")
    
    # Train-test split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
    )
    
    # Scale features
    print(f"\nScaling features...")
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)
    
    # Train Random Forest
    print(f"Training Random Forest...")
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_split=20,
        min_samples_leaf=10,
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    
    model.fit(X_tr_scaled, y_tr)
    
    # Evaluate
    print(f"\n{'='*80}")
    print("MODEL EVALUATION")
    print(f"{'='*80}")
    
    y_pred = model.predict(X_te_scaled)
    acc = accuracy_score(y_te, y_pred)
    
    print(f"\nTest Accuracy: {acc:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_te, y_pred, 
                                target_names=['ice', 'water', 'other'],
                                labels=[1, 2, 3]))
    
    # Feature importance
    print(f"\nFeature Importance:")
    feature_names = ['R', 'G', 'B', 'R_mean', 'G_mean', 'B_mean', 
                    'Brightness', 'B/R ratio', 'G/R ratio']
    importances = model.feature_importances_
    for name, imp in sorted(zip(feature_names, importances), 
                           key=lambda x: x[1], reverse=True):
        print(f"  {name:12s}: {imp:.4f}")
    
    return model, scaler

## Step 6: Segmenting Images Function

In [29]:
def segment_image(image_path, model, scaler, output_path=None):
    """
    Segment a single image
    """
    # Load image
    image = np.array(Image.open(image_path))
    h, w = image.shape[:2]
    
    # Extract features
    features = extract_features_fast(image)
    X = features.reshape(-1, features.shape[-1])
    
    # Scale and predict
    X_scaled = scaler.transform(X)
    predictions = model.predict(X_scaled)
    
    # Reshape to image
    segmentation = predictions.reshape(h, w)
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(image)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Color map: ice=cyan, water=blue, other=gray
    colors = np.array([[0, 0, 0],      # 0: black (unused)
                      [0, 255, 255],   # 1: cyan (ice)
                      [0, 0, 255],     # 2: blue (water)
                      [128, 128, 128]]) # 3: gray (other)
    
    seg_colored = colors[segmentation]
    axes[1].imshow(seg_colored.astype(np.uint8))
    axes[1].set_title('Segmentation')
    axes[1].axis('off')
    
    # Overlay
    overlay = image.copy()
    overlay = (overlay * 0.6 + seg_colored * 0.4).astype(np.uint8)
    axes[2].imshow(overlay)
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {output_path}")
    else:
        plt.show()
    
    plt.close()
    
    return segmentation


In [39]:
def segment_all_images(image_dir, model, scaler, output_dir):
    """
    Segment all images in a directory
    """
    os.makedirs(output_dir, exist_ok=True)
    
    image_paths = glob(os.path.join(image_dir, '*.jpg')) + \
                  glob(os.path.join(image_dir, '*.png')) + \
                  glob(os.path.join(image_dir, '*.tif')) + \
                  glob(os.path.join(image_dir, '*.tiff'))
    
    print(f"\n{'='*80}")
    print(f"BATCH SEGMENTATION")
    print(f"{'='*80}")
    print(f"Found {len(image_paths)} images")
    
    if len(image_paths) == 0:
        print("⚠️  No images found! Check your IMAGE_DIR path.")
        return
    
    for i, img_path in enumerate(image_paths):
        print(f"Segmenting {i+1}/{len(image_paths)}: {os.path.basename(img_path)}")
        
        # Create output filename (change extension to .png for visualization)
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        output_path = os.path.join(output_dir, f"seg_{base_name}.png")
        
        try:
            segmentation = segment_image(img_path, model, scaler, output_path)
            
            # Save raw segmentation as numpy
            seg_path = os.path.join(output_dir, f"seg_{base_name}.npy")
            np.save(seg_path, segmentation)
            
            print(f"  ✓ Saved: {base_name}.png and {base_name}.npy")
            
        except Exception as e:
            print(f"  ✗ ERROR: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    print(f"\n✅ Segmentation complete! Results in: {output_dir}")


## Step 7: Pipeline For Training The Model

In [40]:
if __name__ == "__main__":
    
    # Configuration
    IMAGE_DIR = '../data/raw/raw_images/'  # <<<< PUT YOUR IMAGE DIRECTORY HERE
    OUTPUT_DIR = '../images/'
    
    print("""
    GLACIER SEGMENTATION PIPELINE
    ==============================
    
    This will:
    1. Load images from '{}'
    2. Train model on first 10 images (using auto-labels)
    3. Segment all images
    4. Save results to '{}'
    
    Classes:
    - Ice/Glacier (cyan)
    - Water/Ocean (blue)
    - Other (gray)
    
    """.format(IMAGE_DIR, OUTPUT_DIR))
    
    # Get image paths (including .tif files)
    image_paths = glob(os.path.join(IMAGE_DIR, '*.jpg')) + \
                  glob(os.path.join(IMAGE_DIR, '*.png')) + \
                  glob(os.path.join(IMAGE_DIR, '*.tif')) + \
                  glob(os.path.join(IMAGE_DIR, '*.tiff'))
    
    if len(image_paths) == 0:
        print(f"⚠️  No images found in {IMAGE_DIR}")
        print("Please put your satellite images in the 'raw_images' folder")
    else:
        print(f"Found {len(image_paths)} images\n")
        
        # Train model
        print("STEP 1: Training model...")
        model, scaler = train_segmentation_model(image_paths, sample_size=5000)
        
        # Save model
        print("\nSaving model...")
        joblib.dump(model, 'glacier_segmentation_model.pkl')
        joblib.dump(scaler, 'glacier_segmentation_scaler.pkl')
        print("✓ Model saved")
        
        # Segment all images
        print("\nSTEP 2: Segmenting all images...")
        segment_all_images(IMAGE_DIR, model, scaler, OUTPUT_DIR)
        
        print("\n" + "="*80)
        print("✅ COMPLETE!")
        print("="*80)
        print(f"\nOutputs:")
        print(f"  - Segmented images: {OUTPUT_DIR}/")
        print(f"  - Model: glacier_segmentation_model.pkl")
        print(f"  - Scaler: glacier_segmentation_scaler.pkl")


    GLACIER SEGMENTATION PIPELINE
    
    This will:
    1. Load images from '../data/raw/raw_images/'
    2. Train model on first 10 images (using auto-labels)
    3. Segment all images
    4. Save results to '../images/'
    
    Classes:
    - Ice/Glacier (cyan)
    - Water/Ocean (blue)
    - Other (gray)
    
    
Found 662 images

STEP 1: Training model...
TRAINING SEGMENTATION MODEL
Processing 1/10: Ronne_North_2018-12-13_S2_RAW.tif
Processing 2/10: Larsen_C_Central_2022-12-11_S2_RAW.tif
Processing 3/10: Totten_Front_2019-11-17_S2_RAW.tif
Processing 4/10: Shackleton_Front_2022-11-18_S2_RAW.tif
Processing 5/10: Totten_Shelf_2022-11-28_S2_RAW.tif
Processing 6/10: Totten_Front_2020-12-11_S2_RAW.tif
Processing 7/10: George_VI_South_2018-12-19_S2_RAW.tif
Processing 8/10: Cook_Front_2023-12-07_S2_RAW.tif
Processing 9/10: George_VI_South_2023-12-02_S2_RAW.tif
Processing 10/10: Larsen_C_North_2020-12-14_S2_RAW.tif

Total training samples: 50,000
Class distribution:
  ice: 31,099 (62.2%